# Install Package

In [ ]:
!pip install kaggle

# Data Preparation for Toxic Text

In [ ]:
import os
import json

# where to store your kaggle.json,according to your Kaggle Environment
kaggle_dir = os.path.expanduser("~/.config/kaggle/")
os.makedirs(kaggle_dir, exist_ok=True)

# your Kaggle API key
kaggle_api = {
    "username": "geronimohe",
    "key": "6fbd8ff41e453aa983090b2e8001419b"
}

kaggle_file = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_file, "w") as f:
    json.dump(kaggle_api, f)

# download dataset for toxic text
target_path = "/data/ningyuanhe/EvasionFromICL/data/toxic_text"

!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge -p {target_path}

In [ ]:
# unzip the downloaded dataset
import zipfile
download_path = "/data/ningyuanhe/EvasionFromICL/data/toxic_text/jigsaw-toxic-comment-classification-challenge.zip"
extract_path = "/data/ningyuanhe/EvasionFromICL/data/toxic_text"

with zipfile.ZipFile(download_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

download_path = "/data/ningyuanhe/EvasionFromICL/data/toxic_text/train.csv.zip"
extract_path = "/data/ningyuanhe/EvasionFromICL/data/toxic_text"
with zipfile.ZipFile(download_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# preprocess the data
# We only use the train dataset rather than the test dataset. This is because the test datset has no labels
# In our experiments, we use column "label" to denote the label name, column "text" to specify the sample text

df = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/train.csv')
columns_to_drop = ['id','severe_toxic','obscene','threat','insult','identity_hate']
df.drop(columns=columns_to_drop, inplace=True)
df = df.rename(columns={'comment_text': 'text'})
df.insert(0, 'idx', range(0, len(df) ))
df['toxic'] = df['toxic'].apply(lambda x: 1 if x == 0 else 0)
df = df.rename(columns={'toxic': 'label_value'})
df.insert(3, 'label','')
df['label'] = df['label_value'].map({1:'benign' , 0: 'toxic'})
data_dir = '/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/'
os.makedirs(data_dir, exist_ok=True)
df.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/normalized_toxic_text_data.csv', index=False)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

#split the data according to label
df = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/normalized_toxic_text_data.csv')
if 'label' in df.columns:
    toxic_data = df[df['label'] == 'toxic']
    benign_data = df[df['label'] == 'benign']
    toxic_data.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/toxic_data.csv', index=False)
    benign_data.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/benign_data.csv', index=False)
    print("successfully split data into toxic_data.csv and benign_data.csv")
else:
    print("the 'label' column don't exist in CSV")

#sample 2500 from the toxic_data and benign_data to reduce the size of the dataset
df = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/toxic_data.csv')
sampled_toxic_data = df.sample(n=2500, random_state=42)  # 将 random_state 设置为 42
sampled_toxic_data.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/sampled_toxic_data.csv', index=False)

df = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/benign_data.csv')
sampled_benign_data = df.sample(n=2500, random_state=42)  # 将 random_state 设置为 42
sampled_benign_data.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/sampled_benign_data.csv', index=False)

#split the data into 80%train and 20%test data
df = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/sampled_toxic_data.csv')
toxic_train_df, toxic_test_df = train_test_split(df, test_size=0.2, random_state=42)
toxic_train_df.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/toxic_train.csv', index=False)
toxic_test_df.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/toxic_test.csv', index=False)

df = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/sampled_benign_data.csv')
benign_train_df, benign_test_df = train_test_split(df, test_size=0.2, random_state=42)
benign_train_df.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/benign_train.csv', index=False)
benign_test_df.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/benign_test.csv', index=False)

#combine the toxic and benign train data into final train data
#combine the toxic and benign test data into final test data
toxic_train = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/toxic_train.csv')
benign_train = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/benign_train.csv')
train_data = pd.concat([toxic_train, benign_train], ignore_index=True)
train_data.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/train.csv', index=False)

toxic_test = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/toxic_test.csv')
benign_test = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/benign_test.csv')
test_data = pd.concat([toxic_test, benign_test], ignore_index=True)
test_data.to_csv('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/test.csv', index=False)

successfully split data into toxic_data.csv and benign_data.csv


# Data Preparation for illicit Promotion

In [ ]:
# preprocess the data
# In our experiments, we use column "label" to denote the label name, column "text" to specify the sample text
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/illicit_promotion/balanced_binary_data.csv')

df.insert(0, 'idx', range(0, len(df) ))
df.drop(columns=['source'], inplace=True)

df['label_value'] = df['label'].map({'benign': 1, 'illicit': 0})

df.insert(df.columns.get_loc('label'), 'label_value', df.pop('label_value'))

df.to_csv('/data/ningyuanhe/EvasionFromICL/data/illicit_promotion/normalized_binary_data.csv', index=False)  

#We split the data into 80% train data and 20% test data
df = pd.read_csv('/data/ningyuanhe/EvasionFromICL/data/illicit_promotion/normalized_binary_data.csv')  

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df.to_csv('/data/ningyuanhe/EvasionFromICL/data/illicit_promotion/train.csv', index=False)  
test_df.to_csv('/data/ningyuanhe/EvasionFromICL/data/illicit_promotion/test.csv', index=False)    